In [ ]:
!pip install matplotlib

In [ ]:
import sys
import time
import matplotlib.pyplot as plt
import numpy as np

# Aumenta o limite de recursão nativo do Python para suportar Ackermann
STACK_LIMIT=100000
sys.setrecursionlimit(STACK_LIMIT)

# MÓDULO 1: FUNÇÕES BÁSICAS INICIAIS

In [ ]:
def Z(x: int) -> int:
    """Função Zero: Z(x) = 0"""
    return 0

def S(x: int) -> int:
    """Função Sucessor: S(x) = x + 1"""
    return x + 1

def P(i: int, n: int, *args) -> int:
    """
    Função Projeção: P_i^n(x_1, ..., x_n) = x_i
    Nota: 'i' usa indexação baseada em 1 (1-indexed) de acordo com a teoria.
    """
    if len(args) != n:
        raise ValueError(f"Esperado {n} argumentos, recebido {len(args)}")
    if i < 1 or i > n:
        raise IndexError(f"Índice i={i} fora dos limites para n={n}")
    return args[i - 1]

# Testes de verificação dos primitivos
print("--- Testes das Funções Básicas ---")
print(f"Z(10) = {Z(10)}")
print(f"S(5) = {S(5)}")
print(f"P_2^3(10, 20, 30) = {P(2, 3, 10, 20, 30)}")

--- Testes das Funções Básicas ---
Z(10) = 0
S(5) = 6
P_2^3(10, 20, 30) = 20


# MÓDULO 2: FUNÇÕES RECURSIVAS PRIMITIVAS (FRP)


In [ ]:
def add_frp(x: int, y: int) -> int:
    """
    Adição definida via FRP:
    add(x, 0) = x
    add(x, y + 1) = S(add(x, y))
    """
    result = x  # Caso base: add(x, 0)
    for _ in range(y):  # Passos de recursão
        result = S(result)
    return result

def mult_frp(x: int, y: int) -> int:
    """
    Multiplicação definida via FRP:
    mult(x, 0) = 0
    mult(x, y + 1) = add(mult(x, y), x)
    """
    result = Z(x)  # Caso base: mult(x, 0)
    for _ in range(y):
        result = add_frp(result, x)
    return result

# Testes das operações FRP
print("--- Testes das Funções Recursivas Primitivas ---")
print(f"add_frp(7, 8) = {add_frp(7, 8)}")
print(f"mult_frp(4, 5) = {mult_frp(4, 5)}")

--- Testes das Funções Recursivas Primitivas ---
add_frp(7, 8) = 15
mult_frp(4, 5) = 20


# MÓDULO 3: ACKERMANN & OPERADOR DE MINIMIZAÇÃO (μ)


In [ ]:
def ackermann(m: int, n: int) -> int:
    """
    Função de Ackermann-Péter:
    A(0, n) = n + 1
    A(m, 0) = A(m - 1, 1)
    A(m, n) = A(m - 1, A(m, n - 1))
    """
    if m == 0:
        return n + 1
    elif n == 0:
        return ackermann(m - 1, 1)
    else:
        return ackermann(m - 1, ackermann(m, n - 1))

def mu_operator(f, target=0):
    """
    Operador de Minimização Não Limitado (μ):
    Retorna o menor y ∈ ℕ tal que f(y) == target.
    Equivale a um laço while indeterminado.
    """
    y = 0
    while True:
        if f(y) == target:
            return y
        y = S(y)

# Testes de Ackermann e Minimização
print("--- Testes da Função de Ackermann ---")
print(f"A(1, 2) = {ackermann(1, 2)}")
print(f"A(2, 2) = {ackermann(2, 2)}")
print(f"A(3, 3) = {ackermann(3, 3)}")
print(f"A(3, 4) = {ackermann(3, 4)}")
# Exemplo do operador mu: busca pela raiz exata (y^2 - x = 0)
x_test = 49
raiz_exata = mu_operator(lambda y: mult_frp(y, y) - x_test)
print(f"\n--- Teste do Operador μ ---")
print(f"Menor y tal que y^2 = {x_test} -> y = {raiz_exata}")

--- Testes da Função de Ackermann ---
A(1, 2) = 4
A(2, 2) = 7
A(3, 3) = 61
A(3, 4) = 125

--- Teste do Operador μ ---
Menor y tal que y^2 = 49 -> y = 7


# MÓDULO 4: RAIZ QUADRADA (FRP vs MINIMIZAÇÃO)


In [ ]:
def sqrt_frp_bounded(x: int) -> int:
    """
    Raiz Quadrada de Piso (FRP):
    Garante parada limitando a busca até x (limite superior razoável).
    Retorna o maior y tal que y^2 <= x.
    """
    ans = 0
    # Como y <= x para todo x >= 0, limitamos o loop finitamente até x+1
    for y in range(x + 1):
        if mult_frp(y, y) <= x:
            ans = y
        else:
            break
    return ans

def sqrt_mu_unbounded(x: int) -> int:
    """
    Raiz Quadrada Inteira Exata / Busca Livre (μ-Recursiva):
    Encontra o menor y tal que (y+1)^2 > x usando busca indeterminada (while).
    """
    def pred(y):
        # f(y) = 0 quando (y+1)^2 > x
        return 0 if mult_frp(S(y), S(y)) > x else 1

    return mu_operator(pred, target=0)

# Testes comparativos
test_val = 20
print(f"Raiz de piso (FRP Bounded) de {test_val}: {sqrt_frp_bounded(test_val)}")
print(f"Raiz de piso (μ Unbounded) de {test_val}: {sqrt_mu_unbounded(test_val)}")

Raiz de piso (FRP Bounded) de 20: 4
Raiz de piso (μ Unbounded) de 20: 4


# MÓDULO 5: BENCHMARK & ANÁLISE DE DESEMPENHO


In [ ]:
def benchmark():
    print("==================================================")
    print("          BENCHMARK DE DESEMPENHO               ")
    print("==================================================\n")

    # 1. Benchmark de Ackermann
    print("[1] Testando Limites de Crescimento da Função de Ackermann:")
    ack_inputs = [(0, 0), (1, 2), (2, 2), (3, 3), (3, 4)]
    for m, n in ack_inputs:
        start = time.perf_counter()
        res = ackermann(m, n)
        elapsed = time.perf_counter() - start
        print(f"  A({m}, {n}) = {res:<10} | Tempo: {elapsed:.6f} segundos")

    # 2. Benchmark FRP vs μ (Raiz Quadrada)
    print("\n[2] Comparando Busca FRP (Limitada) vs Busca μ (Minimização):")
    numeros = [100, 1000, 5000]

    for n in numeros:
        # FRP
        t0 = time.perf_counter()
        r_frp = sqrt_frp_bounded(n)
        t_frp = time.perf_counter() - t0

        # μ-Operator
        t0 = time.perf_counter()
        r_mu = sqrt_mu_unbounded(n)
        t_mu = time.perf_counter() - t0

        print(f"  Número: {n:<5}")
        print(f"    - FRP Bounded  : resultado={r_frp}, tempo={t_frp:.6f}s")
        print(f"    - μ Unbounded  : resultado={r_mu}, tempo={t_mu:.6f}s")

benchmark()

          BENCHMARK DE DESEMPENHO               

[1] Testando Limites de Crescimento da Função de Ackermann:
  A(0, 0) = 1          | Tempo: 0.000001 segundos
  A(1, 2) = 4          | Tempo: 0.000001 segundos
  A(2, 2) = 7          | Tempo: 0.000003 segundos
  A(3, 3) = 61         | Tempo: 0.000151 segundos
  A(3, 4) = 125        | Tempo: 0.005183 segundos

[2] Comparando Busca FRP (Limitada) vs Busca μ (Minimização):
  Número: 100  
    - FRP Bounded  : resultado=10, tempo=0.000047s
    - μ Unbounded  : resultado=10, tempo=0.000046s
  Número: 1000 
    - FRP Bounded  : resultado=31, tempo=0.002757s
    - μ Unbounded  : resultado=31, tempo=0.004742s
  Número: 5000 
    - FRP Bounded  : resultado=70, tempo=0.025815s
    - μ Unbounded  : resultado=70, tempo=0.022646s


# MÓDULO 6: GRÁFICO DE CONSUMO DA PILHA (ACKERMANN)


In [ ]:
def testar_limites_ackermann_v3(m_max: int, n_max: int, stack_limit: int):
    sys.setrecursionlimit(stack_limit)

    casos = []
    picos_pilha = []
    estourou = False

    # Contador direto de profundidade (Instantâneo, O(1), sem overhead de I/O)
    def ack_depth_fast(m, n, depth=1):
        nonlocal max_d
        if depth > max_d:
            max_d = depth

        if m == 0:
            return n + 1
        elif n == 0:
            return ack_depth_fast(m - 1, 1, depth + 1)
        else:
            return ack_depth_fast(m - 1, ack_depth_fast(m, n - 1, depth + 1), depth + 1)

    # Entradas organizadas
    entradas_teste = [(m, n) for m in range(m_max + 1) for n in range(n_max + 1)]

    for m, n in entradas_teste:
        if estourou:
            break

        max_d = 0
        rotulo = f"$A({m}, {n})$"

        try:
            ack_depth_fast(m, n)
            casos.append(rotulo)
            picos_pilha.append(max_d)
        except RecursionError:
            casos.append(f"{rotulo}*")
            picos_pilha.append(stack_limit)
            estourou = True

    # --- PLOTAGEM FORMATADA E SEM ENCAVALAMENTO ---
    plt.figure(figsize=(12, 6))

    cores = ['#3498db' if p < stack_limit else '#e74c3c' for p in picos_pilha]
    bars = plt.bar(casos, picos_pilha, color=cores, width=0.55)

    plt.yscale('log')
    plt.axhline(y=stack_limit, color='#2c3e50', linestyle='--', linewidth=1.5, label=f'Limite da Pilha ({stack_limit})')

    # Ajustes estéticos no eixo X (45 graus de rotação)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.title(f'Pico Máximo de Profundidade da Pilha (Limite: {stack_limit})', fontsize=12, fontweight='bold', pad=15)
    plt.xlabel('Entrada $A(m, n)$ (* = Estouro de Pilha / Crash)', fontsize=10, labelpad=10)
    plt.ylabel('Profundidade Máxima de Chamadas (Escala Log10)', fontsize=10)
    plt.grid(True, which="both", ls=":", alpha=0.4)

    # Espaço dinâmico no topo do gráfico para evitar corte dos valores
    plt.ylim(bottom=0.8, top=stack_limit * 4)

    # Adiciona os valores no topo das barras
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval * 1.25, f'{yval:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    plt.legend(loc='upper left', frameon=True)
    plt.tight_layout()
    plt.show()

# PAINEL DE CONFIGURAÇÃO (Execução instantânea)
testar_limites_ackermann_v3(m_max=4, n_max=2, stack_limit=STACK_LIMIT)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_13629/2742568271.py", line 69, in <cell line: 0>
    testar_limites_ackermann_v3(m_max=4, n_max=2, stack_limit=STACK_LIMIT)
  File "/tmp/ipykernel_13629/2742568271.py", line 32, in testar_limites_ackermann_v3
    ack_depth_fast(m, n)
  File "/tmp/ipykernel_13629/2742568271.py", line 19, in ack_depth_fast
    return ack_depth_fast(m - 1, ack_depth_fast(m, n - 1, depth + 1), depth + 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13629/2742568271.py", line 19, in ack_depth_fast
    return ack_depth_fast(m - 1, ack_depth_fast(m, n - 1, depth + 1), depth + 1)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13629/2742568271.py", line 19, in ack_depth_fast
    return ack_depth

TypeError: object of type 'NoneType' has no len()